# 課題と解答例：10_pde_diffusion_basics

元Notebook: [../10_pde_diffusion_basics.ipynb](../10_pde_diffusion_basics.ipynb)

## 5. 課題

1. `dt` を変え，どの値から振動が増えるか調べ，理論上の上限と比較する．
2. `nx` を201にして `dx` を半分にする．安定性を保つには `dt` を何倍にすべきか確かめる．
3. Neumann条件を `u_new[0] = u_new[-1] = 0` というDirichlet条件へ変え，総量と長時間後の分布がどう変わるか説明する．
4. 高さが同じで幅の異なる2つの初期分布を比較し，ピーク値と総量のどちらを評価指標にすべきか考える．

### 追加実装課題

5. `run_diffusion_experiment(dt_factor, boundary, width)` という関数を作り，`r`，安定条件を満たすか，初期総量，最終総量，最終最大値を辞書で返す．
6. `dt_factor` を0.25，0.45，0.55，0.70に変えた表を作り，どの条件で数値不安定性が出るかを表から説明する．
7. `boundary="neumann"` と `boundary="dirichlet"` を切り替えられるようにし，総量の変化を同じ表に入れる．

## 解答例

1. このNotebookでは `D = 0.1`, `nx = 101` なので，`dx = 0.01` である．安定条件は

   ```{math}
   r=\frac{D\,dt}{dx^2}\le \frac12
   ```

   であるから，

   ```{math}
   dt\le \frac{dx^2}{2D}=\frac{0.01^2}{2\times0.1}=0.0005
   ```

   が理論上の上限である．`dt = 0.45 * dx**2 / D` では `r = 0.45` なので安定であり，`dt = 0.70 * dx**2 / D` では `r = 0.70` なので振動が増える．

2. `nx` を201にすると，区間長が同じなら `dx` はほぼ半分になる．安定条件では `dt` は `dx**2` に比例するので，安定性を保つには `dt` を約4分の1にする必要がある．格子を2倍細かくすると，時間刻みは4倍厳しくなる．

3. Neumann条件では端を通る流束が0なので，総量はほぼ保存される．一方，`u_new[0] = u_new[-1] = 0` とすると端の値を0に固定するDirichlet条件になる．端が吸収境界のように働くため，総量は減少し，長時間後には全体が0に近づく．

4. 高さが同じで幅が異なる初期分布では，広い分布ほど総量が大きい．ピーク値だけを見ると同じ初期条件に見えてしまうが，拡散で保存される量は面積に対応する総量である．したがって，物質量や熱量を比較するなら総量を評価指標にする．一方，最大温度や局所的な危険度を問うならピーク値を評価指標にしてよい．

5. 実装例である．

   ```python
   def initial_condition_width(x, width):
       return np.exp(-((x - 0.5) / width)**2)

   def step_with_boundary(u, r, boundary):
       u_new = u.copy()
       u_new[1:-1] = u[1:-1] + r * (u[:-2] - 2*u[1:-1] + u[2:])
       if boundary == "neumann":
           u_new[0] = u_new[1]
           u_new[-1] = u_new[-2]
       elif boundary == "dirichlet":
           u_new[0] = 0.0
           u_new[-1] = 0.0
       else:
           raise ValueError(boundary)
       return u_new

   def run_diffusion_experiment(dt_factor, boundary="neumann", width=0.08):
       dt_try = dt_factor * dx**2 / D
       r_try = D * dt_try / dx**2
       u = initial_condition_width(x, width)
       mass0 = np.trapz(u, x)
       for _ in range(int(np.ceil(t_end / dt_try))):
           u = step_with_boundary(u, r_try, boundary)
       return {
           "dt_factor": dt_factor,
           "boundary": boundary,
           "r": r_try,
           "stable": r_try <= 0.5,
           "mass_initial": mass0,
           "mass_final": np.trapz(u, x),
           "max_final": u.max(),
       }
   ```

6. 表は次のように作れる．

   ```python
   rows = [run_diffusion_experiment(f, "neumann") for f in [0.25, 0.45, 0.55, 0.70]]
   pd.DataFrame(rows)
   ```

   `r > 0.5` の行で数値不安定性が出やすい．

7. 境界条件の比較は，`boundary` を変えて同じ関数を呼び出す．

   ```python
   rows = []
   for boundary in ["neumann", "dirichlet"]:
       for f in [0.25, 0.45]:
           rows.append(run_diffusion_experiment(f, boundary))
   pd.DataFrame(rows)
   ```

   Neumann条件では総量が保存されやすく，Dirichlet条件では端から量が失われる．